# paper-verdict-finetune -- Colab GPU runner

Before running: Runtime -> Change runtime type -> GPU.

This notebook is intentionally thin: it clones the repo, installs
requirements, and calls the scripts in `src/`. All logic lives there.

**Persistence model** (so a run can be launched, closed, and reviewed
later -- Colab's local disk does not survive a runtime reset):
- **Checkpoints + full per-run outputs** -> uploaded to a Hugging Face
  Hub model repo, under `runs/<run_id>/`. Weights are too large to put
  in git on every run, and this keeps it off your Google Drive entirely
  (no risk to anything else stored there) -- temporary third-party
  storage, not tied to this project.
- **Lightweight run record** (config, metrics, confusion matrix -- no
  weights) -> committed and pushed back to this branch, so `RUNS.md` in
  the repo is always a current, append-only index of every run, readable
  from any future session without a Hugging Face token or downloading
  anything. The Hub repo holds the heavy artifact; GitHub holds the map
  of what exists and what it scored.

In [ ]:
!nvidia-smi

In [ ]:
import os
from huggingface_hub import login, create_repo

# paste a Hugging Face token with WRITE access (huggingface.co/settings/tokens)
# input() keeps it out of the notebook's saved output/history
HF_TOKEN = input("Hugging Face token (write access): ").strip()
login(token=HF_TOKEN)

# a model repo dedicated to holding run checkpoints -- replace with your
# own username/repo-name once, then reuse it across every run; created
# automatically (private) the first time if it doesn't exist yet
HF_REPO_ID = "YOUR_HF_USERNAME/paper-verdict-finetune-runs"
create_repo(HF_REPO_ID, private=True, exist_ok=True)
os.environ["HF_REPO_ID"] = HF_REPO_ID

In [ ]:
BRANCH = "verdict-finetune-architecture"
!git clone --branch $BRANCH --depth 1 https://github.com/AngryWatership/textor.git repo
%cd repo
!git config user.email "noreply@anthropic.com"
!git config user.name "Colab runner"

In [ ]:
!pip install -q -r requirements.txt huggingface_hub

## Review past runs before launching a new one

`RUNS.md` came in with the clone -- it's the current index of everything
already tried. Look at this before picking the next config, so you're
not re-running something already done, and so you know what the current
best is to try to beat.

In [ ]:
print(open("RUNS.md").read())

In [ ]:
# sanity-check the split before committing GPU time to training
!python src/data.py --data data/papers.jsonl --split-out outputs/split.json

In [ ]:
import sys
sys.path.insert(0, "src")
from runlog import new_run_id

BASE_MODEL = "allenai/specter2_base"  # or sentence-transformers/all-MiniLM-L6-v2, allenai/scibert_scivocab_uncased
FINETUNE_MODE = "partial"  # or "full"

RUN_ID = new_run_id(BASE_MODEL, FINETUNE_MODE)
print("RUN_ID:", RUN_ID)
os.environ["RUN_ID"] = RUN_ID
os.environ["BASE_MODEL"] = BASE_MODEL
os.environ["FINETUNE_MODE"] = FINETUNE_MODE

In [ ]:
!cd src && python train.py \
    --base-model $BASE_MODEL \
    --finetune-mode $FINETUNE_MODE \
    --run-id $RUN_ID \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs \
    --repo-root ..

In [ ]:
!cd src && python evaluate.py \
    --run-id $RUN_ID \
    --base-model $BASE_MODEL \
    --finetune-mode $FINETUNE_MODE \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs \
    --repo-root ..

In [ ]:
import json
report = json.load(open(f"outputs/{RUN_ID}/eval_report.json"))
print("confusion matrix (rows=true, cols=pred), labels:", report["confusion_matrix_labels"])
for row in report["confusion_matrix"]:
    print(row)
print("test silhouette:", report["test_silhouette_euclidean"])
print("test knn10 purity:", report["test_knn10_purity"])
print("per-class purity:", report["test_knn10_purity_per_class"])
print("\ncompare against docs/prior_results.md and RUNS.md for every other run so far.")

## Persist: full run (incl. weights) to Hugging Face Hub, lightweight record to GitHub

In [ ]:
from huggingface_hub import upload_folder

hub_run_path = f"runs/{RUN_ID}"
upload_folder(
    repo_id=HF_REPO_ID,
    folder_path=f"outputs/{RUN_ID}",
    path_in_repo=hub_run_path,
    commit_message=f"Upload run {RUN_ID}",
)
checkpoint_url = f"https://huggingface.co/{HF_REPO_ID}/resolve/main/{hub_run_path}/best_model.pt"
print("uploaded full run (including best_model.pt) to", checkpoint_url)

# record the Hub path in the log, then regenerate RUNS.md with it, so a
# later session knows exactly where to find the checkpoint without having
# downloaded it
from runlog import upsert_run
upsert_run(".", RUN_ID, {"checkpoint_path": checkpoint_url})

In [ ]:
# push the lightweight record (RUNS.md, runs_log.jsonl, this run's config/
# train_summary/eval_report -- NOT best_model.pt, that only lives on the
# Hugging Face Hub) back to the branch, so it's reviewable from any future
# session without Colab or a Hub token at all.
!git add RUNS.md runs_log.jsonl outputs/$RUN_ID/config.json outputs/$RUN_ID/train_summary.json outputs/$RUN_ID/eval_report.json
!git commit -m "Log run $RUN_ID" || echo "nothing new to commit"
!git push origin $BRANCH

## To resume / review in a later session

- `git clone` this branch -> `RUNS.md` shows every run and its metrics,
  no Hugging Face token or Colab needed to see *what happened*.
- To re-load an actual checkpoint for further use, download it from the
  `checkpoint_path` URL logged in `RUNS.md` / `runs_log.jsonl` (a
  Hugging Face Hub file URL), then point `evaluate.py --checkpoint` at
  the downloaded file.